Evaluation uses datasets, parameter values and fit records already produced
by the library. There is no single evaluation protocol: each comparison
requires the representation and assumptions of the helper being used.

## Which objects answer which questions?

| Question | Library inputs and operations | Complete procedure |
|---|---|---|
| Does the model reproduce a behavioral pattern? | Observed and simulated `RecallDataset` objects passed to the same analysis and plotting functions, such as `plot_spc()`. | [Simulation](../templates/simulation.ipynb) |
| What changes when a parameter changes? | Shared study inputs, a fixed model configuration and deliberate parameter interventions followed by simulation and analysis. | [Parameter exploration](../templates/parameter_shifting.ipynb) |
| How do likelihood fits compare? | Compatible named fit records passed to comparison helpers in `jaxcmr.summarize`. | [Model comparison](../templates/model_comparison.ipynb) |

A behavioral match, an intervention and a likelihood comparison answer
different questions. In particular, forced agreement with the observed
recall length is not evidence that a model predicts when recall stops.

## Fit-record conventions at the comparison boundary

`FitResult` provides parameter roles, objective values and row identifiers.
Comparison helpers additionally use a `name`. The templates record the data
selection, model configuration and objective needed to establish comparability.

Before comparing likelihood scores, establish that the records concern the
same observations, included events and likelihood definition. Check whether
parameters were pooled or independently fitted and how many values were
estimated. A lower discrepancy loss cannot be substituted for NLL merely
because both appear in a field named `fitness`.

The supplied source-only and broader eCMR records are pooled fits to the same
all-condition selection with the same termination-excluded objective and
stopping policy. Their emotional modulation differs in whether it also acts
on temporal context. Each has ten free parameters and one pooled fit row.

These examples were refitted under the current library. Both learn before
context updating, fix emotional encoding drift at 1 and tie emotional recall
drift to temporal recall drift. They omit the legacy `emotion_drift_rate`.
Historical fits also require their original context equations and parameter
interpretations; a shared import path alone does not identify that behavior.

In [1]:
#| code-summary: "Read the named pooled records and their comparison basis"

from pathlib import Path
import json

files = {
    "Source-only eCMR": "Dupertuys2026_eCMR_source_only.json",
    "Broader eCMR": "Dupertuys2026_eCMR_broad.json",
}
records = [
    {**json.loads((Path("data/fits") / filename).read_text()), "name": name}
    for name, filename in files.items()
]
for record in records:
    print(record["name"], "— fit IDs:", record["fits"]["subject"],
          "free parameters:", len(record["free"]))
print("Recorded trial selection:", records[0]["trial_query"])
print("Recorded loss:", records[0]["loss_function"])

Source-only eCMR — fit IDs: [-1] free parameters: 10
Broader eCMR — fit IDs: [-1] free parameters: 10
Recorded trial selection: data['list_type'] > 0
Recorded loss: jaxcmr.loss.transform_sequence_likelihood.ExcludeTerminationLikelihoodLoss


## Supply the objective convention a helper expects

`jaxcmr.summarize.calculate_aic()` accepts a list of dictionaries containing
`name`, `free` and `fitness`, and returns a DataFrame sorted by AIC. It uses
the length of `free` as the parameter count and **sums `fitness` as log
likelihood**:

$$\mathrm{AIC} = 2k - 2\log L = 2k + 2\,\mathrm{NLL}.$$

These saved fits contain NLL, so create comparison views with negated
`fitness` values. The original fit records retain their original objective.

In [2]:
#| code-summary: "Adapt NLL records to the library AIC helper"

from jaxcmr.summarize import calculate_aic

log_likelihood_records = [
    {**record, "fitness": [-value for value in record["fitness"]]}
    for record in records
]
calculate_aic(log_likelihood_records)

,Model,AIC
1,Broader eCMR,14348.715746
0,Source-only eCMR,14358.852540


The new source-only and broader fits have NLLs 7169.426 and 7164.358,
respectively. Their AIC difference is 10.137, favoring broader eCMR.
The within-model differences between the two three-restart seed sets are
0.005 and 0.049 NLL units.
The ranking is the same in both seed sets.

The result is a library-produced table. Lower AIC is preferred within this
comparison; it does not establish a unique explanation of the experiment.
Each model contributes one pooled parameter set, so `len(free)` is the
appropriate count here. Independently fitted participant rows require
counting the parameters estimated across those fits; this pooled example
should not be applied unchanged to them.

Check each helper's objective convention. For example,
`calculate_aic_weights()` currently consumes NLL, unlike `calculate_aic()`.
The log-likelihood views above are specifically for this call. Paired
participant comparisons also require genuinely participant-level fit rows;
a pooled row is not a sample of independent participant estimates.

## Experimental cross-validation functionality

`jaxcmr.experimental.cross_validation` contains an experimental driver, with
`CVResult` describing fold-specific fits, participant identifiers, training
scores and held-out scores. The current driver mixes pooled fitting with
participant-level scoring and can return incorrect held-out scores; it is
not ready for scientific use. A held-out
comparison requires an appropriate split, fitting only the training
observations and evaluating the held-out observations under the chosen
objective. It is a different procedure from comparing in-sample AIC values.

See [Fitting Procedures](fitting.ipynb) for result structure and the
[model comparison template](../templates/model_comparison.ipynb) for the full
pooled comparison. These interfaces organize the evidence; selecting a
scientifically meaningful comparison remains part of the research design.